# Is that difference real?

MichAl Academy, lesson 1.9.

Run each cell with **Shift+Enter**.

Here we know the right answer in advance, which is the one luxury a simulation
gives you and real life never does. Detector A really catches 70% of attacks and
detector B really catches 73%. B is better, by exactly three points.

Now watch how hard that is to see.

## 1. One test set

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

TRUE_A, TRUE_B = 0.70, 0.73


def run_test(n, rng):
    """Run both detectors over n attacks. Returns how many each caught."""
    return rng.binomial(n, TRUE_A), rng.binomial(n, TRUE_B)


for _ in range(6):
    ka, kb = run_test(50, rng)
    print(f"A caught {ka:3d}/50 ({ka/50:5.1%})   B caught {kb:3d}/50 ({kb/50:5.1%})   "
          f"measured gap {(kb - ka) / 50:+6.1%}")

The true gap is +3.0 points every single time. Look at what the measurements
say. Some of them have A ahead.

## 2. The interval

A confidence interval says: given what this test set showed, the gap is
plausibly somewhere in here.

In [ ]:
def gap_interval(k_a, k_b, n, z=1.96):
    pa, pb = k_a / n, k_b / n
    se = np.sqrt(pa * (1 - pa) / n + pb * (1 - pb) / n)
    diff = pb - pa
    return diff, diff - z * se, diff + z * se


for n in [50, 200, 1000, 4000]:
    ka, kb = run_test(n, rng)
    diff, lo, hi = gap_interval(ka, kb, n)
    verdict = "B is ahead" if lo > 0 else "cannot tell"
    print(f"n={n:5d}   gap {diff:+6.1%}   plausibly {lo:+6.1%} to {hi:+6.1%}   {verdict}")

The number in the middle is not the useful part. **Whether the range contains
zero** is the useful part. If it does, "B is better" is a sentence this data does
not support, whatever the middle number says.

## 3. How often does a test set get it wrong?

One run tells you nothing about how much a run can lie. Twenty thousand do.

In [ ]:
TRIALS = 20_000

print(f"{'n':>6} {'coverage':>10} {'found it':>10} {'wrong sign':>12}")
for n in [50, 100, 200, 500, 1000, 2000, 4000]:
    ka = rng.binomial(n, TRUE_A, TRIALS)
    kb = rng.binomial(n, TRUE_B, TRIALS)
    diff, lo, hi = gap_interval(ka, kb, n)

    coverage   = ((lo <= 0.03) & (hi >= 0.03)).mean()   # does the range contain the truth?
    found      = (lo > 0).mean()                        # did it detect a real difference?
    wrong_sign = (diff < 0).mean()                      # did it point the wrong way?

    print(f"{n:>6} {coverage:>10.1%} {found:>10.1%} {wrong_sign:>12.1%}")

Three things in that table.

**Coverage sits at about 95% everywhere.** That is the interval doing its job.
It is not more accurate at large n, it is *narrower*.

**"Found it" is the honest measure of a test set's power.** At n=50 almost no
test set can detect a real three-point difference. You would run the comparison,
see a gap, and have learned nothing.

**"Wrong sign" is the one that should worry you.** At n=50 roughly a third of
test sets say A is ahead. A is not ahead. If you ship on one of those you have
made a real decision on pure noise.

## 4. Four times the data to halve the uncertainty

Look at the shape of the standard error: `n` is on the bottom, inside a square
root.

In [ ]:
import matplotlib.pyplot as plt

ns = np.arange(20, 5001, 20)
p = 0.72
se = np.sqrt(2 * p * (1 - p) / ns)
width = 2 * 1.96 * se           # full width of the 95% interval

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(ns, width * 100)
ax.axhline(3.0, color="red", linestyle="--", label="the real gap, 3 points")
ax.set_xlabel("test set size")
ax.set_ylabel("width of the 95% range, points")
ax.legend()
plt.show()

for n in [250, 1000, 4000]:
    w = 2 * 1.96 * np.sqrt(2 * p * (1 - p) / n) * 100
    print(f"n={n:5d}   interval is {w:5.2f} points wide")

Quadrupling the data halves the width. That is why going from 50 to 100 rows
does not help and going to 2,000 does.

## 5. A bootstrap, for when there is no formula

The interval above has a formula because accuracy is a proportion. Precision,
recall, F1 and AUC do not have convenient ones.

The bootstrap needs no formula at all: resample your test set with replacement
many times, recompute the metric each time, and look at the spread of answers.

In [ ]:
# One fixed test set: which attacks each detector caught, row by row
n = 400
caught_a = rng.random(n) < TRUE_A
caught_b = rng.random(n) < TRUE_B

observed = caught_b.mean() - caught_a.mean()

idx = rng.integers(0, n, size=(10_000, n))          # 10,000 resamples of the same rows
boot = caught_b[idx].mean(axis=1) - caught_a[idx].mean(axis=1)

lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"observed gap  {observed:+.1%}")
print(f"bootstrap 95% {lo:+.1%} to {hi:+.1%}")
print("contains zero?", lo <= 0 <= hi)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(boot * 100, bins=50)
ax.axvline(0, color="red", linestyle="--", label="no difference")
ax.set_xlabel("gap in points, across 10,000 resamples")
ax.legend()
plt.show()

Same idea, no algebra, and it works for any metric you can compute. This is the
tool to reach for in Track 2 when the metric is precision at a fixed alert
budget and no textbook has a formula for it.

## 6. Your turn

A colleague reports that the retrained model is better:

- old model: 912 correct out of 1,000
- new model: 928 correct out of 1,000

Is that improvement supported by the test set? And if not, how big a test set
would settle it?

In [ ]:
K_OLD, K_NEW, N = 912, 928, 1000

diff, lo, hi = gap_interval(K_OLD, K_NEW, N)

print(f"measured improvement {diff:+.1%}")
print(f"plausibly            {lo:+.1%} to {hi:+.1%}")
print()
print("supported?", lo > 0)

In [ ]:
# TODO: find the smallest test set that would make a gap this size conclusive,
# assuming the true rates really are 91.2% and 92.8%.
needed = None

for n in range(500, 20_001, 100):
    pass    # TODO: work out the interval at this n and stop when it clears zero

print("rows needed:", needed)

At each `n`, the interval is centred on the same 1.6 point gap and gets narrower.
You want the first `n` where the bottom of it clears zero.

<details>
<summary>Answer</summary>

The improvement is not supported. The plausible range runs from about -0.8 to
+4.0 points, so "no difference at all" is well inside it.

```python
needed = None
for n in range(500, 20_001, 100):
    _, lo, _ = gap_interval(round(0.912 * n), round(0.928 * n), n)
    if lo > 0:
        needed = n
        break
```

That lands around 2,200 rows, better than double what was tested on. The
uncomfortable part is that nothing was done wrong here: the model may well be
better. The test set simply cannot say so, and the write-up said it anyway.

The right sentence is "1.6 points better, plausibly anywhere from slightly worse
to four points better, on 1,000 rows". Nobody makes a bad decision from that.

</details>

## What you now have

- A measured difference is a real difference plus noise, and small test sets are mostly noise
- Report the interval, not the point. Whether it contains zero is the question
- Standard error has `n` under a square root: four times the data to halve the uncertainty
- At small `n` a real difference frequently measures the wrong way round
- The bootstrap gives you an interval for any metric, with no formula
- None of this helps if the test set is not a fair sample, and in security it usually is not: your labels are the attacks that got caught

Next is lesson 1.10, gradients, and then Track 1 is done.